# Lab 12 · Vẽ hình cho báo cáo và kiểm tra bằng `assert`

**Lập trình xử lý dữ liệu · 2627-1 · Giờ thực hành · Bài 12**

Chọn **Save a copy in Drive** trước khi sửa.

Lab đi theo bài giảng Bài 12. Bạn sẽ vẽ ba hình bằng `fig, ax`: biểu đồ đường theo thời gian,
thanh ngang xếp hạng quận và histogram. Mỗi hình có tiêu đề nêu thông điệp, nhãn trục có đơn vị
và được lưu vào `figures/` bằng code. Sau đó bạn sửa một biểu đồ cột có trục y bị cắt.
Các ô kiểm tra dùng `assert` để kiểm tra cấu trúc của hình, như với mọi kết quả khác.

**Giáo trình:** [McKinney, chương 9, §9.1–9.2](https://wesmckinney.com/book/plotting-and-visualization).
Lab dùng thêm `isin` (Bài 4), `groupby` và `agg` (Bài 5), `resample` và cách cắt theo mốc chụp (Bài 8).

## Cách làm việc trong bài lab

- Điền các ô `TODO` và phần giải thích.
  `assert` kiểm tra một số điều kiện, không thay thế việc xem các trường hợp sai.
- Với các phần có hướng dẫn, tự viết trước khi nhờ AI.
  Các bài kiểm tra trên lớp (🚫 đóng) đánh giá các kỹ năng này.
- Bài tự làm ✅ mở cho phép dùng AI, kèm khai báo và cách kiểm chứng.
- Khi cần hỗ trợ, trao đổi với giảng viên thực hành về yêu cầu hoặc trường hợp chưa hiểu.

## Mục tiêu

1. Vẽ biểu đồ đường, thanh ngang và histogram bằng `fig, ax`, với tiêu đề nêu thông điệp,
   nhãn trục có đơn vị và nguồn.
2. Lưu hình vào `figures/` bằng `savefig`, để hình sinh lại được bằng code.
3. Kiểm tra cấu trúc hình bằng `assert`: tiêu đề, nhãn trục, số cột, trục bắt đầu từ 0, file đã lưu.
4. Chọn `bins` để histogram cho thấy hình dạng của phân phối.
5. Sửa một biểu đồ cột có trục y bị cắt; rà soát hình bằng năm điểm kiểm tra của Bài 12.

## Chuẩn bị

Dữ liệu thật là Santiago, mốc chụp **29/06/2026**, hai file `visualisations/`:

- `listings.csv`: mỗi hàng là một chỗ ở; `id` là mã chỗ ở, `neighbourhood` là quận, `room_type` là loại phòng;
  `availability_365` là số ngày còn trống trong 365 ngày tới, tính từ mốc chụp.
- `reviews.csv`: mỗi hàng là một đánh giá, gồm `listing_id` (mã chỗ ở) và `date`.

Dùng file cục bộ nếu có; nếu chưa có thì đọc đúng mốc chụp từ nguồn.
Ô chuẩn bị cũng tạo thư mục `figures/` và khai báo `rcParams` một lần cho mọi hình
(slide «Định dạng chung cho các hình» ở Bài 12). Hàm `tieu_de(ax)` lấy tiêu đề của khung,
kể cả tiêu đề đặt bằng `loc="left"` như bài giảng, hoặc tiêu đề của cả hình (`fig.suptitle`);
các ô kiểm tra dùng hàm này.

Chạy hai ô dưới trước. Nếu phiên Colab đã nạp pandas phiên bản khác, chọn
**Restart session and run all** sau khi cài.

In [ ]:
%pip -q install pandas==3.0.3

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BASE = ("https://data.insideairbnb.com/chile/rm/santiago/"
        "2026-06-29/visualisations/")
tep_ls = Path("data/listings_santiago_2026-06-29.csv")
tep_rv = Path("data/reviews_santiago_2026-06-29.csv")
df = pd.read_csv(tep_ls if tep_ls.exists() else BASE + "listings.csv")
rv = pd.read_csv(tep_rv if tep_rv.exists() else BASE + "reviews.csv",
                 parse_dates=["date"])

Path("figures").mkdir(exist_ok=True)   # nơi lưu hình xuất ra
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 11})
XANH, CAM = "#1E93AB", "#E8890C"


def tieu_de(ax):
    """Tiêu đề của khung ax (kể cả loc="left"/"right") và của cả hình (fig.suptitle)."""
    chu = " ".join(ax.get_title(loc=l) for l in ("left", "center", "right"))
    return (chu + " " + ax.figure.get_suptitle()).strip()


print("pandas:", pd.__version__)
print("Số chỗ ở:", len(df), "| Số đánh giá:", len(rv))

## Phần 1 · Số đánh giá theo tháng ở Providencia

Câu hỏi: số đánh giá mỗi tháng ở quận Providencia thay đổi thế nào?
`rv` không có cột quận, nên cần biết mỗi đánh giá thuộc chỗ ở nào: lấy mã các chỗ ở thuộc Providencia
trong `df`, rồi lọc `rv` theo `listing_id` bằng `isin` (Bài 4).

**a)** Tạo `ids_pro` (mã các chỗ ở thuộc Providencia) và `thang_pro`: số đánh giá mỗi tháng của các chỗ ở đó
(`resample("ME").size()`, Bài 8). Chỉ giữ đánh giá ghi ngày đến mốc chụp 29/06/2026, rồi bỏ tháng 06/2026
chưa trọn bằng nhãn (`.loc[:"2026-05"]`), không cắt theo vị trí.

In [ ]:
# TODO: tạo ids_pro và thang_pro.
ids_pro = None
thang_pro = None

In [ ]:
assert ids_pro is not None and thang_pro is not None, "Tạo ids_pro và thang_pro."
assert isinstance(thang_pro, pd.Series), 'thang_pro là Series: dùng resample("ME").size().'
assert len(thang_pro) > 0, (
    "Không còn đánh giá nào sau khi lọc: ids_pro phải là cột id (mã chỗ ở) của các chỗ ở thuộc Providencia.")
assert len(ids_pro) == 2997, "ids_pro gồm mã của mọi chỗ ở có neighbourhood là Providencia."
assert thang_pro.index.max() == pd.Timestamp("2026-05-31"), (
    "Tháng cuối là 05/2026: lọc date <= 2026-06-29, rồi bỏ tháng 06/2026 chưa trọn bằng nhãn.")
assert len(thang_pro) == 187, 'Đếm theo tháng bằng resample("ME"), từ tháng đầu tiên có đánh giá.'
assert thang_pro.iloc[-1] == 3777, "Chỉ đếm đánh giá của các chỗ ở thuộc Providencia."
print(f"{len(thang_pro)} tháng, {thang_pro.index.min():%m/%Y} đến {thang_pro.index.max():%m/%Y}; "
      f"tháng 05/2026: {thang_pro.iloc[-1]:,} đánh giá.")

**b)** Vẽ biểu đồ đường từ `thang_pro` và lưu vào `figures/providencia_thang.png`:

- `fig, ax = plt.subplots(figsize=(9, 3.5))`, rồi `ax.plot(...)`;
- tiêu đề nêu thông điệp, không chỉ gọi tên biểu đồ (có thể đặt bằng `loc="left"` như bài giảng);
- nhãn trục y có đơn vị;
- nguồn và mốc chụp bằng `ax.text(..., transform=ax.transAxes)`, như hình «Thành phần của một biểu đồ» ở Bài 12;
- `fig.savefig("figures/providencia_thang.png", dpi=150, bbox_inches="tight")`.

Dòng đầu của ô dưới xoá file của lần chạy trước (nếu có), để phần kiểm tra chỉ nhận file vừa lưu.

In [ ]:
Path("figures/providencia_thang.png").unlink(missing_ok=True)   # xoá hình của lần chạy trước

# TODO: vẽ và lưu hình.
ax = None

In [ ]:
assert ax is not None, "Tạo fig, ax bằng plt.subplots rồi vẽ."
assert any(len(l.get_xdata()) == len(thang_pro) and max(l.get_ydata()) == thang_pro.max()
           for l in ax.lines), f"Vẽ một đường từ thang_pro: {len(thang_pro)} điểm, mỗi tháng một điểm."
assert len(tieu_de(ax)) >= 15, "Tiêu đề phải nêu một thông điệp, không chỉ gọi tên biểu đồ."
assert ax.get_ylabel() != "", "Thiếu nhãn trục y (có đơn vị)."
assert any("nguồn" in t.get_text().lower() and t.get_transform() == ax.transAxes for t in ax.texts), (
    'Ghi nguồn bằng ax.text(x, y, "Nguồn: ...", transform=ax.transAxes).')
assert Path("figures/providencia_thang.png").exists(), (
    "Lưu hình bằng fig.savefig vào figures/providencia_thang.png.")
print("Hình 1 đạt các điều kiện cấu trúc; mở thư mục figures/ để xem file PNG.")

**Nhận xét của bạn:** Số đánh giá ở Providencia thay đổi thế nào từ 2022 đến 05/2026? Năm 2020 có gì khác
trên hình? Tiêu đề của bạn có nói nhiều hơn điều hình cho thấy không? …

## Phần 2 · Tỷ lệ nguyên căn theo quận

Câu hỏi: trong các quận có nhiều chỗ ở, quận nào có tỷ lệ nguyên căn (`room_type == "Entire home/apt"`) cao nhất?

**a)** Ô dưới tạo cột boolean `nguyen_can`. Tạo `tk`: mỗi quận một hàng (chỉ mục là tên quận), hai cột

- `n`: số chỗ ở của quận (`size`), kể cả chỗ ở thiếu giá;
- `pct`: tỷ lệ nguyên căn, **dạng tỷ lệ từ 0 đến 1** (trung bình của cột boolean).

Tạo `tk8`: các quận có `n` từ 300 trở lên, sắp **tăng dần** theo `pct`, để thanh dài nhất nằm trên cùng
của biểu đồ thanh ngang.

In [ ]:
df["nguyen_can"] = df["room_type"] == "Entire home/apt"

# TODO: tạo tk và tk8.
tk = None
tk8 = None

In [ ]:
assert tk is not None and tk8 is not None, "Tạo tk và tk8."
assert isinstance(tk8, pd.DataFrame), "tk8 là DataFrame có hai cột n và pct."
assert "Lo Barnechea" in tk8.index, "Chỉ mục của tk là tên quận (groupby không dùng as_index=False)."
assert {"n", "pct"} <= set(tk8.columns), "tk8 cần hai cột n và pct."
assert tk8["pct"].between(0, 1).all(), (
    "pct là tỷ lệ từ 0 đến 1 (trung bình của cột boolean); chỉ nhân 100 khi vẽ.")
assert len(tk8) == 8 and tk8["n"].min() >= 300, "Giữ các quận có từ 300 chỗ ở trở lên (8 quận)."
assert tk8["n"].sum() == 16_850, "n là số chỗ ở của quận (size), kể cả chỗ ở thiếu giá."
assert tk8["pct"].is_monotonic_increasing, "Sắp tk8 tăng dần theo pct."
assert round(tk8["pct"].iloc[-1], 3) == 0.899, "Quận có pct cao nhất là Lo Barnechea, khoảng 0,899."
print((tk8["pct"] * 100).round(1))

**b)** Vẽ biểu đồ thanh ngang từ `tk8`, độ dài thanh tính theo **%** (`pct * 100`). Ghi giá trị ở đầu mỗi thanh
bằng `ax.bar_label(bars, fmt="%.0f%%")`, đặt tiêu đề nêu thông điệp và nhãn trục x có đơn vị %,
rồi lưu vào `figures/pct_nguyen_can.png`.

In [ ]:
Path("figures/pct_nguyen_can.png").unlink(missing_ok=True)   # xoá hình của lần chạy trước

# TODO: vẽ và lưu hình.
ax = None

In [ ]:
assert ax is not None, "Tạo fig, ax bằng plt.subplots rồi vẽ."
rong = [p.get_width() for p in ax.patches]
assert len(rong) == 8, "Vẽ đúng 8 thanh ngang (ax.barh), mỗi quận một thanh."
assert np.allclose(rong, tk8["pct"] * 100), (
    "Vẽ thanh ngang (ax.barh); độ dài thanh là pct * 100 (đơn vị %), theo đúng thứ tự tăng dần của tk8.")
assert ax.get_xlim()[0] == 0, "Trục x của biểu đồ thanh phải bắt đầu từ 0."
assert len(ax.texts) >= 8, 'Ghi giá trị ở đầu mỗi thanh bằng ax.bar_label(bars, fmt="%.0f%%").'
assert len(tieu_de(ax)) >= 15, "Tiêu đề phải nêu một thông điệp, không chỉ gọi tên biểu đồ."
assert "%" in ax.get_xlabel(), "Nhãn trục x cần ghi đơn vị %."
assert Path("figures/pct_nguyen_can.png").exists(), "Lưu hình bằng fig.savefig vào figures/pct_nguyen_can.png."
print("Hình 2 đạt: 8 thanh theo %, trục từ 0, thanh dài nhất ở trên cùng.")

**Nhận xét của bạn:** Hình chỉ gồm các quận có từ 300 chỗ ở. Vì sao cần ngưỡng này, và người đọc
biết ngưỡng đó ở đâu trên hình của bạn? …

## Phần 3 · Phân phối số ngày còn trống

`availability_365` là số ngày còn trống trong 365 ngày tới, tính từ mốc chụp.
Trung bình hay trung vị không cho biết hình dạng của phân phối; histogram cho biết.

Vẽ histogram `availability_365` của mọi chỗ ở trong `df`, **mỗi cột rộng 5 ngày**, từ 0 đến 365
(`bins=range(0, 370, 5)`). Đặt tiêu đề nêu điều bạn thấy trên hình, nhãn cho cả hai trục,
rồi lưu vào `figures/availability_hist.png`.

In [ ]:
Path("figures/availability_hist.png").unlink(missing_ok=True)   # xoá hình của lần chạy trước

# TODO: vẽ và lưu histogram.
ax = None

In [ ]:
assert ax is not None, "Tạo fig, ax bằng plt.subplots rồi vẽ."
cot = ax.patches
assert len(cot) == 73 and np.allclose([p.get_width() for p in cot], 5), (
    "Mỗi cột rộng 5 ngày, từ 0 đến 365: bins=range(0, 370, 5) cho 73 cột.")
assert sum(p.get_height() for p in cot) == len(df), (
    "Tổng chiều cao các cột phải bằng số chỗ ở: vẽ availability_365 của cả df, trục y là số chỗ ở.")
assert len(tieu_de(ax)) >= 15, "Tiêu đề phải nêu một thông điệp, không chỉ gọi tên biểu đồ."
assert ax.get_xlabel() != "" and ax.get_ylabel() != "", "Đặt nhãn cho cả hai trục."
assert Path("figures/availability_hist.png").exists(), (
    "Lưu hình bằng fig.savefig vào figures/availability_hist.png.")
a = df["availability_365"]
print(f"trung vị: {a.median():.0f} ngày | 0 ngày: {(a == 0).sum()} chỗ ở | "
      f"365 ngày: {(a == 365).sum():,} | từ 360 ngày: {(a >= 360).sum():,}")

Phân phối dồn về phía 365 ngày (còn trống gần trọn 365 ngày tới): trung vị là 321 ngày; 3.789 chỗ ở còn trống từ 360 ngày
trở lên, riêng 365 ngày có 1.759 chỗ ở. Ngoài ra có các đỉnh nhọn: ở 0, với 469 chỗ ở không còn ngày trống
(đã kín lịch hoặc chủ nhà chặn lịch), và quanh 90, 180, 270 ngày (89 ngày: 201 chỗ ở; 269 ngày: 353 chỗ ở).
Một con số trung vị không cho thấy các đỉnh này.

**Nhận xét của bạn:** Vẽ lại với `bins=10`: các đỉnh quanh 90, 180, 270 ngày còn thấy không?
Các đỉnh này gợi ý điều gì về cách chủ nhà mở lịch, và cần thêm dữ liệu gì để kiểm chứng?
Vì sao không nên đọc `availability_365 == 0` là "chỗ ở đông khách"? …

## Phần 4 · Sửa biểu đồ có trục y bị cắt

Ô dưới dùng số liệu đúng: tỷ lệ nguyên căn của bốn quận, **tính theo %** (đã nhân 100).
Trục y bắt đầu từ 70 nên chênh lệch trông lớn hơn thực tế. Chạy ô để xem hình.

In [ ]:
pct = (df.groupby("neighbourhood")["nguyen_can"].mean() * 100).loc[
    ["Providencia", "Vitacura", "La Florida", "Las Condes"]].sort_values()
print(pct.round(1))

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(pct.index, pct.values, color=CAM)
ax.set_ylim(70, 85)
ax.set_title("Vitacura thấp hơn hẳn Las Condes")
plt.show()

Vẽ lại bằng biểu đồ cột từ `pct`: trục y bắt đầu từ 0, nhãn trục y có đơn vị %,
tiêu đề mới nêu đúng mức chênh lệch.

In [ ]:
# TODO: vẽ bản sửa.
ax = None

In [ ]:
assert ax is not None, "Tạo fig, ax bằng plt.subplots rồi vẽ."
chieu_cao = sorted(p.get_height() for p in ax.patches)
assert len(chieu_cao) == 4, "Vẽ đúng 4 cột từ pct."
assert np.allclose(chieu_cao, sorted(pct.values)), (
    "Chiều cao cột là pct (đơn vị %, đã nhân 100), không phải tỷ lệ 0–1.")
assert ax.get_ylim()[0] == 0, "Biểu đồ cột: trục y phải bắt đầu từ 0."
assert ax.get_ylim()[1] >= pct.max(), "Trục y phải đủ cao để thấy trọn các cột."
assert ax.get_ylabel() != "", "Thiếu nhãn trục y (đơn vị %)."
assert len(tieu_de(ax)) >= 15 and tieu_de(ax) != "Vitacura thấp hơn hẳn Las Condes", (
    "Đặt tiêu đề mới, nêu đúng mức chênh lệch.")
thap, cao = pct.iloc[0], pct.iloc[-1]
print(f"{pct.index[0]} {thap:.1f}%, {pct.index[-1]} {cao:.1f}%: chênh {cao - thap:.1f} điểm phần trăm.")
print(f"Cột {pct.index[-1]} dài gấp cột {pct.index[0]}: {(cao - 70) / (thap - 70):.1f} lần khi trục từ 70; "
      f"{cao / thap:.2f} lần khi trục từ 0.")

**Nhận xét của bạn:** Bốn quận có tỷ lệ nguyên căn từ 72,6% đến 82,3%, chênh nhau khoảng 10 điểm phần trăm.
Chênh lệch đó có đáng nêu trong báo cáo không, và cần thêm thông tin gì (vd cỡ mẫu n của từng quận) để kết luận?
Điểm kiểm tra nào trong năm điểm phát hiện được lỗi của hình gốc? …

## Phần 5 · Bài tự làm ✅ mở (làm sớm tại lớp hoặc làm tại nhà)

### Tự làm 1 · Bộ hình cho thành phố của nhóm

Lặp lại Phần 1–3 với **thành phố trong bài tập lớn** của nhóm: đổi `BASE` sang mốc chụp của thành phố đó
và chọn một quận cho Phần 1. Mỗi hình có tiêu đề nêu thông điệp, nhãn trục có đơn vị, nguồn và được lưu vào
`figures/`; rà soát từng hình bằng năm điểm kiểm tra: thông điệp; trục; cỡ mẫu; dạng và màu; nguồn và diễn giải.
Viết hai câu diễn giải dưới mỗi hình.

In [ ]:
# Viết bài tự làm 1 ở đây

**Nhận xét của bạn:** …

### Tự làm 2 · Rà soát một hình do AI vẽ

Nhờ một công cụ AI viết code vẽ hình ở Phần 2 (tỷ lệ nguyên căn của các quận có từ 300 chỗ ở) chỉ từ mô tả bằng lời,
không đưa code của bạn. Chạy code đó rồi rà soát hình bằng năm điểm kiểm tra. Ghi lại điểm nào chưa đạt,
và tự tính lại ít nhất một giá trị trên hình (vd thanh của Lo Barnechea, so với `tk.loc["Lo Barnechea"]`).

In [ ]:
# Viết bài tự làm 2 ở đây

**Nhận xét của bạn:** …

**Khai báo AI (nếu dùng):** công cụ, yêu cầu chính, một kết quả bạn đã tự kiểm tra. …

## Tóm tắt bài lab

- Mỗi hình trả lời một câu hỏi: đường cho diễn biến theo thời gian, thanh ngang cho xếp hạng nhóm,
  histogram cho phân phối.
- Tiêu đề nêu thông điệp; trục có nhãn và đơn vị; nguồn và mốc chụp ghi ngay trên hình.
- Hình lưu vào `figures/` bằng `savefig` để sinh lại được bằng code. `assert` kiểm tra cấu trúc hình,
  không thay cho việc nhìn hình.
- Chọn `bins` theo đơn vị của dữ liệu (ở đây 5 ngày) để thấy hình dạng phân phối; một con số trung vị
  không cho thấy các đỉnh.
- Biểu đồ cột bắt đầu từ 0; trục y bị cắt phóng đại chênh lệch.

Chạy **Restart session and run all**, rồi kiểm tra: mọi ô code đều có kết quả; các ô
nhận xét và khai báo AI (nếu dùng) đã điền đủ.
Trên Colab, thư mục `figures/` nằm trong bộ nhớ phiên và mất khi phiên kết thúc; tải file về nếu cần giữ.

**Đọc thêm:** [McKinney chương 9](https://wesmckinney.com/book/plotting-and-visualization),
[matplotlib: Anatomy of a figure](https://matplotlib.org/stable/gallery/showcase/anatomy.html).

**Bài sau:** trực quan hoá nâng cao: seaborn, bản đồ theo quận và phản biện biểu đồ.